In [1]:
##importando pacotes
import pandas as pd
import numpy as np
import pandas_gbq
from google.cloud import bigquery
import re
import calendar ##importando essa biblioteca pra ter acesso a mes/dias/anos
import locale ##pra configurar o idioma do calendario
import glob
import openpyxl
import shutil
import zipfile

c:\Users\ana.sales_republica\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
client = bigquery.Client()


In [3]:
query = '''SELECT * FROM `repositoriodedadosgpsp.portal_transparencia_cgu.2024_mar_siape_servidores_cadastro`'''

In [4]:
df = pandas_gbq.read_gbq(query, project_id='repositoriodedadosgpsp')
df

Downloading: 100%|██████████|


,Id_SERVIDOR_PORTAL,NOME,CPF,MATRICULA,DESCRICAO_CARGO,CLASSE_CARGO,REFERENCIA_CARGO,PADRAO_CARGO,NIVEL_CARGO,SIGLA_FUNCAO,...,DATA_NOMEACAO_CARGOFUNCAO,DATA_INGRESSO_ORGAO,DOCUMENTO_INGRESSO_SERVICOPUBLICO,DATA_DIPLOMA_INGRESSO_SERVICOPUBLICO,DIPLOMA_INGRESSO_CARGOFUNCAO,DIPLOMA_INGRESSO_ORGAO,DIPLOMA_INGRESSO_SERVICOPUBLICO,UF_EXERCICIO,ANO,MES
0,2722726,RODRIGO DA COSTA FONSECA,***.148.041-**,010****,CONSELHEIRO,None,0.0,None,0.0,-1,...,NaN,20/08/1992,000000S/N,23/11/1984,NaN,PORTARIA,PORTARIA,-1,2024,3
1,2916915,WILMAR LACERDA,***.001.561-**,012****,TECNICO-A,A,0.0,None,16.0,-1,...,NaN,29/05/1979,SN,29/05/1979,NaN,CONTRATO,CONTRATO,-1,2024,3
2,140185,CELIO FARIA JUNIOR,***.194.281-**,021****,ECONOMISTA,S,0.0,II,0.0,-1,...,NaN,21/09/2006,000870_94,21/12/1994,NaN,PORTARIA,PORTARIA,-1,2024,3
3,1605783,VANESSA FERREIRA DE LIMA,***.849.131-**,013****,ECONOMISTA,C,0.0,V,0.0,-1,...,NaN,01/01/2023,77,24/01/2006,NaN,DECRETO,PORTARIA,-1,2024,3
4,2854116,THAISA GOIS FARIAS DE MOURA SANTOS LIMA,***.555.724-**,014****,ENFERMEIRO,C,0.0,IV,0.0,-1,...,NaN,10/04/2006,159,08/05/2006,NaN,PORTARIA,PORTARIA,-1,2024,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
714918,1394764,UDSON AUGUSTO LIMA SANTOS,***.933.911-**,016****,ASSISTENTE EM CIENCIA E TECNOLOGIA,S,0.0,IV,0.0,-1,...,NaN,02/06/2014,55,11/06/2014,NaN,PORTARIA,PORTARIA,-1,2024,3
714919,3056056,URBANO PASCOAL DE OLIVEIRA JUNIOR,***.285.235-**,020****,ASSISTENTE EM CIENCIA E TECNOLOGIA,S,0.0,V,0.0,-1,...,NaN,22/07/2013,95,05/08/2013,NaN,PORTARIA,PORTARIA,-1,2024,3
714920,158382,VINICIUS BORGES MIATELO,***.725.101-**,018****,ASSISTENTE EM CIENCIA E TECNOLOGIA,S,0.0,V,0.0,-1,...,NaN,26/07/2018,105,19/08/2013,NaN,PORTARIA,PORTARIA,-1,2024,3
714921,1328284,GERMANO DE OLIVEIRA FARIAS,***.088.067-**,010****,AUDITOR FEDERAL DE FINANCAS E CONTROLE,S,0.0,IV,0.0,-1,...,NaN,22/11/2016,000000606,30/12/1994,NaN,PORTARIA,PORTARIA,-1,2024,3


In [62]:
df['ORG_LOTACAO'].unique()

array(['Colégio Pedro II', 'Instituto Federal do Paraná',
       'Universidade Federal do Piauí',
       'Fundação Universidade de Brasília',
       'Universidade Federal de Santa Catarina', 'Comando da Marinha',
       'ADVOCACIA-GERAL DA UNIAO', 'MIN GESTAO E INOV EM SERV PUBLICOS',
       'Controladoria-Geral da União', 'Fundação Osório',
       'Comando do Exército', 'MINISTERIO DA DEFESA',
       'Ministério da Saúde', 'Universidade Federal do Rio Grande do Sul',
       'Instituto Brasileiro de Museus',
       'Universidade Federal da Paraíba',
       'Universidade Federal de Mato Grosso',
       'Instituto Federal de Roraima',
       'Universidade Federal de Minas Gerais',
       'Universidade Federal de Santa Maria - RS',
       'Universidade Federal Fluminense - RJ',
       'Universidade Federal do Rio de Janeiro',
       'Instituto Federal do Acre',
       'Fundação Universidade Federal do Amazonas',
       'Instituto Federal do Amazonas',
       'Universidade Federal de Campi

In [ ]:
MGI - Órgão setorial > MIN GESTAO E INOV EM SERV PUBLICOS
Agência Nacional de Petróleo, Gás Natural e Biocombustíveis (ANP) > Agência Nacional do Petróleo	Gás Natural e Biocombustíveis [ testar]
Agência Nacional de Transportes Terrestres (ANTT) > Agência Nacional de Transportes Terrestres
Ministério da Previdência Social (MPS) > MINISTERIO DA PREVIDENCIA SOCIAL e Ministério da Previdência Social
Ministério de Desenvolvimento Social e Combate à Fome (MDS) > MIN DESENV ASSIS SOCI FAMIL COMBATE FOME
Ministério do Meio Ambiente e Mudança do Clima (MMA) > MINISTERIO DO MEIO AMBIENTE
Ministério de Portos e Aeroportos (MPOR) > MINISTERIO DE PORTOS E AEROPORTOS
Ministério do Empreendedorismo (MEMP) > MIN EMPREEND MICROEMP EMP PEQUENO PORTE


In [7]:

mapeamento = {
    "MIN GESTAO E INOV EM SERV PUBLICOS": "MGI - Órgão setorial",
    "Agência Nacional do Petróleo\tGás Natural e Biocombustíveis [ testar]": "Agência Nacional de Petróleo, Gás Natural e Biocombustíveis (ANP)",
    "Agência Nacional de Transportes Terrestres": "Agência Nacional de Transportes Terrestres (ANTT)",
    "MINISTERIO DA PREVIDENCIA SOCIAL": "Ministério da Previdência Social (MPS)",
    "Ministério da Previdência Social": "Ministério da Previdência Social (MPS)",
    "MIN DESENV ASSIS SOCI FAMIL COMBATE FOME": "Ministério de Desenvolvimento Social e Combate à Fome (MDS)",
    "MINISTERIO DO MEIO AMBIENTE": "Ministério do Meio Ambiente e Mudança do Clima (MMA)",
    "MINISTERIO DE PORTOS E AEROPORTOS": "Ministério de Portos e Aeroportos (MPOR)",
    "MIN EMPREEND MICROEMP EMP PEQUENO PORTE": "Ministério do Empreendedorismo (MEMP)"
}




In [12]:
df = df[df['SITUACAO_VINCULO'] == 'ATIVO PERMANENTE']

In [17]:
df

,Id_SERVIDOR_PORTAL,NOME,CPF,MATRICULA,DESCRICAO_CARGO,CLASSE_CARGO,REFERENCIA_CARGO,PADRAO_CARGO,NIVEL_CARGO,SIGLA_FUNCAO,...,DATA_INGRESSO_ORGAO,DOCUMENTO_INGRESSO_SERVICOPUBLICO,DATA_DIPLOMA_INGRESSO_SERVICOPUBLICO,DIPLOMA_INGRESSO_CARGOFUNCAO,DIPLOMA_INGRESSO_ORGAO,DIPLOMA_INGRESSO_SERVICOPUBLICO,UF_EXERCICIO,ANO,MES,ORG_PADRONIZADO
66,1040644,DIEGO ALVES AUGUSTO MARINS,***.743.367-**,021****,AUDITOR,E,0.0,407,0.0,-1,...,14/04/2014,2664,14/04/2014,NaN,PORTARIA,PORTARIA,-1,2024,3,Colégio Pedro II
67,1149929,CLAUDIO ANTONIO COSTA DE BARROS,***.497.587-**,016****,CONTADOR,E,0.0,411,0.0,-1,...,23/12/2008,2016,13/01/2009,NaN,PORTARIA,PORTARIA,-1,2024,3,Colégio Pedro II
68,1371705,CRISTINA ALVES PEREIRA,***.737.987-**,017****,CONTADOR,E,0.0,410,0.0,-1,...,28/09/2009,1726,28/09/2009,NaN,PORTARIA,PORTARIA,-1,2024,3,Colégio Pedro II
69,374940,FABIO COSTA DE ALMEIDA,***.003.207-**,021****,CONTADOR,E,0.0,407,0.0,-1,...,17/03/2014,963,17/03/2014,NaN,PORTARIA,PORTARIA,-1,2024,3,Colégio Pedro II
70,1064625,FELLIPE SANTOS COELHO,***.491.127-**,021****,CONTADOR,E,0.0,407,0.0,-1,...,13/03/2014,964,17/03/2014,NaN,PORTARIA,PORTARIA,-1,2024,3,Colégio Pedro II
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
714916,2936633,THIAGO CARVALHO MARTINS,***.317.121-**,017****,ASSISTENTE EM CIENCIA E TECNOLOGIA,S,0.0,V,0.0,-1,...,23/08/2013,1175,11/01/2010,NaN,PORTARIA,PORTARIA,-1,2024,3,Fundação Coordenação de Aperfeiçoamento de Pes...
714917,1150414,THINAMI SAIKI,***.050.911-**,017****,ASSISTENTE EM CIENCIA E TECNOLOGIA,R,0.0,III,0.0,-1,...,05/05/2009,038,25/05/2009,NaN,PORTARIA,PORTARIA,-1,2024,3,Fundação Coordenação de Aperfeiçoamento de Pes...
714918,1394764,UDSON AUGUSTO LIMA SANTOS,***.933.911-**,016****,ASSISTENTE EM CIENCIA E TECNOLOGIA,S,0.0,IV,0.0,-1,...,02/06/2014,55,11/06/2014,NaN,PORTARIA,PORTARIA,-1,2024,3,Fundação Coordenação de Aperfeiçoamento de Pes...
714919,3056056,URBANO PASCOAL DE OLIVEIRA JUNIOR,***.285.235-**,020****,ASSISTENTE EM CIENCIA E TECNOLOGIA,S,0.0,V,0.0,-1,...,22/07/2013,95,05/08/2013,NaN,PORTARIA,PORTARIA,-1,2024,3,Fundação Coordenação de Aperfeiçoamento de Pes...


In [95]:
df['Id_SERVIDOR_PORTAL'].duplicated().sum()


75929

In [99]:
df_unique['CPF'].duplicated().sum()


78409

In [ ]:
df_unique = df.drop_duplicates(subset=['Id_SERVIDOR_PORTAL'])


In [20]:
df_unique

,Id_SERVIDOR_PORTAL,NOME,CPF,MATRICULA,DESCRICAO_CARGO,CLASSE_CARGO,REFERENCIA_CARGO,PADRAO_CARGO,NIVEL_CARGO,SIGLA_FUNCAO,...,DATA_INGRESSO_ORGAO,DOCUMENTO_INGRESSO_SERVICOPUBLICO,DATA_DIPLOMA_INGRESSO_SERVICOPUBLICO,DIPLOMA_INGRESSO_CARGOFUNCAO,DIPLOMA_INGRESSO_ORGAO,DIPLOMA_INGRESSO_SERVICOPUBLICO,UF_EXERCICIO,ANO,MES,ORG_PADRONIZADO
66,1040644,DIEGO ALVES AUGUSTO MARINS,***.743.367-**,021****,AUDITOR,E,0.0,407,0.0,-1,...,14/04/2014,2664,14/04/2014,NaN,PORTARIA,PORTARIA,-1,2024,3,Colégio Pedro II
67,1149929,CLAUDIO ANTONIO COSTA DE BARROS,***.497.587-**,016****,CONTADOR,E,0.0,411,0.0,-1,...,23/12/2008,2016,13/01/2009,NaN,PORTARIA,PORTARIA,-1,2024,3,Colégio Pedro II
68,1371705,CRISTINA ALVES PEREIRA,***.737.987-**,017****,CONTADOR,E,0.0,410,0.0,-1,...,28/09/2009,1726,28/09/2009,NaN,PORTARIA,PORTARIA,-1,2024,3,Colégio Pedro II
69,374940,FABIO COSTA DE ALMEIDA,***.003.207-**,021****,CONTADOR,E,0.0,407,0.0,-1,...,17/03/2014,963,17/03/2014,NaN,PORTARIA,PORTARIA,-1,2024,3,Colégio Pedro II
70,1064625,FELLIPE SANTOS COELHO,***.491.127-**,021****,CONTADOR,E,0.0,407,0.0,-1,...,13/03/2014,964,17/03/2014,NaN,PORTARIA,PORTARIA,-1,2024,3,Colégio Pedro II
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
714913,262971,THAINARA SILVA ALMEIDA,***.346.605-**,020****,ASSISTENTE EM CIENCIA E TECNOLOGIA,S,0.0,V,0.0,-1,...,23/08/2013,114,09/09/2013,NaN,PORTARIA,PORTARIA,-1,2024,3,Fundação Coordenação de Aperfeiçoamento de Pes...
714915,2445139,THAIS SAUTCHUK PIMENTA,***.082.591-**,016****,ASSISTENTE EM CIENCIA E TECNOLOGIA,R,0.0,III,0.0,-1,...,05/05/2009,38,25/05/2009,NaN,PORTARIA,PORTARIA,-1,2024,3,Fundação Coordenação de Aperfeiçoamento de Pes...
714916,2936633,THIAGO CARVALHO MARTINS,***.317.121-**,017****,ASSISTENTE EM CIENCIA E TECNOLOGIA,S,0.0,V,0.0,-1,...,23/08/2013,1175,11/01/2010,NaN,PORTARIA,PORTARIA,-1,2024,3,Fundação Coordenação de Aperfeiçoamento de Pes...
714917,1150414,THINAMI SAIKI,***.050.911-**,017****,ASSISTENTE EM CIENCIA E TECNOLOGIA,R,0.0,III,0.0,-1,...,05/05/2009,038,25/05/2009,NaN,PORTARIA,PORTARIA,-1,2024,3,Fundação Coordenação de Aperfeiçoamento de Pes...


In [21]:
total_servidores = len(df_unique)
print(f"Total de servidores únicos: {total_servidores}")

Total de servidores únicos: 406192


In [44]:
mapeamento = {
    "MIN GESTAO E INOV EM SERV PUBLICOS": "MGI - Órgão setorial",
    "Agência Nacional do Petróleo\tGás Natural e Biocombustíveis": "Agência Nacional de Petróleo, Gás Natural e Biocombustíveis (ANP)",
    "Agência Nacional de Transportes Terrestres": "Agência Nacional de Transportes Terrestres (ANTT)",
    "MINISTERIO DA PREVIDENCIA SOCIAL": "Ministério da Previdência Social (MPS)",
    "Ministério da Previdência Social": "Ministério da Previdência Social (MPS)",
    "MIN DESENV ASSIS SOCI FAMIL COMBATE FOME": "Ministério de Desenvolvimento Social e Combate à Fome (MDS)",
    "MINISTERIO DO MEIO AMBIENTE": "Ministério do Meio Ambiente e Mudança do Clima (MMA)",
    "MINISTERIO DE PORTOS E AEROPORTOS": "Ministério de Portos e Aeroportos (MPOR)",
    "MIN EMPREEND MICROEMP EMP PEQUENO PORTE": "Ministério do Empreendedorismo (MEMP)"
}

# cria nova coluna padronizada
df_unique["ORG_PADRONIZADO"] = df_unique["ORG_LOTACAO"].replace(mapeamento)

# total por órgão
orgaos = df_unique["ORG_PADRONIZADO"].value_counts()

C:\Users\ana.sales_republica\AppData\Local\Temp\ipykernel_8112\3670785565.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_unique["ORG_PADRONIZADO"] = df_unique["ORG_LOTACAO"].replace(mapeamento)


In [46]:
orgaos

ORG_PADRONIZADO
MINISTERIO DA FAZENDA                                  19694
Instituto Nacional do Seguro Social                    18434
Ministério da Saúde                                    17104
Universidade Federal do Rio de Janeiro                 11828
Universidade Federal de Minas Gerais                    6972
                                                       ...  
Defensoria Pública da União                                5
MINISTERIO DA IGUALDADE RACIAL                             2
MINISTERIO DAS MULHERES                                    2
Ministério da Infraestrutura                               1
Ministério da Agricultura, Pecuária e Abastecimento        1
Name: count, Length: 203, dtype: int64

In [47]:
df_unique['ORG_PADRONIZADO'] = df_unique['ORG_LOTACAO'].replace(mapeamento)


C:\Users\ana.sales_republica\AppData\Local\Temp\ipykernel_8112\3563933449.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_unique['ORG_PADRONIZADO'] = df_unique['ORG_LOTACAO'].replace(mapeamento)


In [49]:
mapeamento = {
    "MIN GESTAO E INOV EM SERV PUBLICOS": "MGI - Órgão setorial",
    "Agência Nacional do Petróleo\tGás Natural e Biocombustíveis": "Agência Nacional de Petróleo, Gás Natural e Biocombustíveis (ANP)",
    "Agência Nacional de Transportes Terrestres": "Agência Nacional de Transportes Terrestres (ANTT)",
    "MINISTERIO DA PREVIDENCIA SOCIAL": "Ministério da Previdência Social (MPS)",
    "Ministério da Previdência Social": "Ministério da Previdência Social (MPS)",
    "MIN DESENV ASSIS SOCI FAMIL COMBATE FOME": "Ministério de Desenvolvimento Social e Combate à Fome (MDS)",
    "MINISTERIO DO MEIO AMBIENTE": "Ministério do Meio Ambiente e Mudança do Clima (MMA)",
    "MINISTERIO DE PORTOS E AEROPORTOS": "Ministério de Portos e Aeroportos (MPOR)",
    "MIN EMPREEND MICROEMP EMP PEQUENO PORTE": "Ministério do Empreendedorismo (MEMP)"
}

In [61]:
df_unique['ORG_LOTACAO'] = df_unique['ORG_LOTACAO'].str.strip().str.replace(r'\s+', ' ', regex=True)

C:\Users\ana.sales_republica\AppData\Local\Temp\ipykernel_8112\4102505828.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_unique['ORG_LOTACAO'] = df_unique['ORG_LOTACAO'].str.strip().str.replace(r'\s+', ' ', regex=True)


In [ ]:
df_unique['ORG_LOTACAO'].unique() MIN DESENV ASSIS SOCI FAMIL COMBATE FOME

Instituto Nacional do Seguro Social
Agência Nacional do Petróleo, Gás Natural e Biocombustíveis

array(['Colégio Pedro II', 'Instituto Federal do Paraná',
       'Universidade Federal do Piauí',
       'Fundação Universidade de Brasília',
       'Universidade Federal de Santa Catarina', 'Comando da Marinha',
       'ADVOCACIA-GERAL DA UNIAO', 'MIN GESTAO E INOV EM SERV PUBLICOS',
       'Controladoria-Geral da União', 'Fundação Osório',
       'Comando do Exército', 'MINISTERIO DA DEFESA',
       'Ministério da Saúde', 'Universidade Federal do Rio Grande do Sul',
       'Instituto Brasileiro de Museus',
       'Universidade Federal da Paraíba',
       'Universidade Federal de Mato Grosso',
       'Instituto Federal de Roraima',
       'Universidade Federal de Minas Gerais',
       'Universidade Federal de Santa Maria - RS',
       'Universidade Federal Fluminense - RJ',
       'Universidade Federal do Rio de Janeiro',
       'Instituto Federal do Acre',
       'Fundação Universidade Federal do Amazonas',
       'Instituto Federal do Amazonas',
       'Universidade Federal de Campi

In [93]:
df_unique

,Id_SERVIDOR_PORTAL,NOME,CPF,MATRICULA,DESCRICAO_CARGO,CLASSE_CARGO,REFERENCIA_CARGO,PADRAO_CARGO,NIVEL_CARGO,SIGLA_FUNCAO,...,DATA_INGRESSO_ORGAO,DOCUMENTO_INGRESSO_SERVICOPUBLICO,DATA_DIPLOMA_INGRESSO_SERVICOPUBLICO,DIPLOMA_INGRESSO_CARGOFUNCAO,DIPLOMA_INGRESSO_ORGAO,DIPLOMA_INGRESSO_SERVICOPUBLICO,UF_EXERCICIO,ANO,MES,ORG_PADRONIZADO
66,1040644,DIEGO ALVES AUGUSTO MARINS,***.743.367-**,021****,AUDITOR,E,0.0,407,0.0,-1,...,14/04/2014,2664,14/04/2014,NaN,PORTARIA,PORTARIA,-1,2024,3,Colégio Pedro II
67,1149929,CLAUDIO ANTONIO COSTA DE BARROS,***.497.587-**,016****,CONTADOR,E,0.0,411,0.0,-1,...,23/12/2008,2016,13/01/2009,NaN,PORTARIA,PORTARIA,-1,2024,3,Colégio Pedro II
68,1371705,CRISTINA ALVES PEREIRA,***.737.987-**,017****,CONTADOR,E,0.0,410,0.0,-1,...,28/09/2009,1726,28/09/2009,NaN,PORTARIA,PORTARIA,-1,2024,3,Colégio Pedro II
69,374940,FABIO COSTA DE ALMEIDA,***.003.207-**,021****,CONTADOR,E,0.0,407,0.0,-1,...,17/03/2014,963,17/03/2014,NaN,PORTARIA,PORTARIA,-1,2024,3,Colégio Pedro II
70,1064625,FELLIPE SANTOS COELHO,***.491.127-**,021****,CONTADOR,E,0.0,407,0.0,-1,...,13/03/2014,964,17/03/2014,NaN,PORTARIA,PORTARIA,-1,2024,3,Colégio Pedro II
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
714913,262971,THAINARA SILVA ALMEIDA,***.346.605-**,020****,ASSISTENTE EM CIENCIA E TECNOLOGIA,S,0.0,V,0.0,-1,...,23/08/2013,114,09/09/2013,NaN,PORTARIA,PORTARIA,-1,2024,3,Fundação Coordenação de Aperfeiçoamento de Pes...
714915,2445139,THAIS SAUTCHUK PIMENTA,***.082.591-**,016****,ASSISTENTE EM CIENCIA E TECNOLOGIA,R,0.0,III,0.0,-1,...,05/05/2009,38,25/05/2009,NaN,PORTARIA,PORTARIA,-1,2024,3,Fundação Coordenação de Aperfeiçoamento de Pes...
714916,2936633,THIAGO CARVALHO MARTINS,***.317.121-**,017****,ASSISTENTE EM CIENCIA E TECNOLOGIA,S,0.0,V,0.0,-1,...,23/08/2013,1175,11/01/2010,NaN,PORTARIA,PORTARIA,-1,2024,3,Fundação Coordenação de Aperfeiçoamento de Pes...
714917,1150414,THINAMI SAIKI,***.050.911-**,017****,ASSISTENTE EM CIENCIA E TECNOLOGIA,R,0.0,III,0.0,-1,...,05/05/2009,038,25/05/2009,NaN,PORTARIA,PORTARIA,-1,2024,3,Fundação Coordenação de Aperfeiçoamento de Pes...


In [ ]:
39168 DE 406192 

In [92]:
12913 + 628 + 882 + 18501 + 213 + 4516 + 1504 + 11

39168

In [100]:
df_unique['CPF'].duplicated().sum()


78409

In [101]:
df_unique["ID_UNICO"] = (
    df_unique["CPF"].astype(str).str.strip() + "_" +
    df_unique["Id_SERVIDOR_PORTAL"].astype(str).str.strip() + "_" +
    df_unique["NOME"].astype(str).str.strip()
)

# contar duplicatas
duplicatas_total = df_unique["ID_UNICO"].duplicated().sum()
print(f"Total de duplicatas: {duplicatas_total}")


Total de duplicatas: 0


C:\Users\ana.sales_republica\AppData\Local\Temp\ipykernel_8112\2083628583.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_unique["ID_UNICO"] = (


In [116]:
duplicatas = df_unique[df_unique["CPF"].duplicated(keep=False)].sort_values("ID_UNICO")
duplicatas

,Id_SERVIDOR_PORTAL,NOME,CPF,MATRICULA,DESCRICAO_CARGO,CLASSE_CARGO,REFERENCIA_CARGO,PADRAO_CARGO,NIVEL_CARGO,SIGLA_FUNCAO,...,DOCUMENTO_INGRESSO_SERVICOPUBLICO,DATA_DIPLOMA_INGRESSO_SERVICOPUBLICO,DIPLOMA_INGRESSO_CARGOFUNCAO,DIPLOMA_INGRESSO_ORGAO,DIPLOMA_INGRESSO_SERVICOPUBLICO,UF_EXERCICIO,ANO,MES,ORG_PADRONIZADO,ID_UNICO
322294,1244835,LISANDRO LUCAS DE LIMA MOURA,***.000.010-**,016****,PROFESSOR ENS BASICO TECN TECNOLOGICO,D,0.0,None,401.0,-1,...,1453,21/10/2010,NaN,PORTARIA,PORTARIA,-1,2024,3,Instituto Federal Sul-rio-grandense,***.000.010-**_1244835_LISANDRO LUCAS DE LIMA ...
487710,1634396,ELOISA MONTANHA SOUZA DA SILVA,***.000.010-**,033****,Sem informaç,None,-1.0,-1,-1.0,FG,...,41,13/04/2023,NaN,PORTARIA,PORTARIA,-1,2024,3,Universidade Federal de Santa Maria - RS,***.000.010-**_1634396_ELOISA MONTANHA SOUZA D...
170451,2463985,FRANCIELE RUSCH KONIG,***.000.010-**,012****,PROFESSOR ENS BASICO TECN TECNOLOGICO,D,0.0,None,101.0,-1,...,1728,25/01/2024,NaN,PORTARIA,PORTARIA,-1,2024,3,Instituto Federal Farroupilha,***.000.010-**_2463985_FRANCIELE RUSCH KONIG
508933,2792695,JOSE LUIS DUARTE RIBEIRO,***.000.010-**,003****,PROFESSOR DO MAGISTERIO SUPERIOR,8,0.0,None,801.0,-1,...,9,10/05/1989,NaN,PORTARIA,PORTARIA,-1,2024,3,Universidade Federal do Rio Grande do Sul,***.000.010-**_2792695_JOSE LUIS DUARTE RIBEIRO
476293,2794677,LORENA GONZALEZ TELIS,***.000.010-**,018****,ASSISTENTE EM ADMINISTRACAO,D,0.0,409,0.0,-1,...,1274,13/09/2010,NaN,PORTARIA,PORTARIA,-1,2024,3,Fundação Universidade Federal do Pampa,***.000.010-**_2794677_LORENA GONZALEZ TELIS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148048,2013282,PAULA EVELINE DA SILVA DOS SANTOS,***.999.877-**,030****,PROFESSOR ENS BASICO TECN TECNOLOGICO,D,0.0,None,301.0,-1,...,1681,28/11/2018,NaN,PORTARIA,PORTARIA,-1,2024,3,Instituto Federal Fluminense,***.999.877-**_2013282_PAULA EVELINE DA SILVA ...
48837,1180548,RENATA SOLANO COUTINHO,***.999.907-**,017****,ENFERMEIRO,C,0.0,III,0.0,-1,...,156,15/03/2010,NaN,PORTARIA,PORTARIA,-1,2024,3,Ministério da Saúde,***.999.907-**_1180548_RENATA SOLANO COUTINHO
55902,361583,BARBARA MARIA CAMPOS MARTINS,***.999.907-**,017****,AGENTE ADMINISTRATIVO,C,0.0,III,0.0,-1,...,MS/1102,07/12/2009,NaN,PORTARIA,PORTARIA,-1,2024,3,Ministério da Saúde,***.999.907-**_361583_BARBARA MARIA CAMPOS MAR...
279687,2034264,PEDRO GOMES DE ALMEIDA FILHO,***.999.934-**,011****,VIGILANTE,D,0.0,316,0.0,-1,...,196,01/01/1995,NaN,PORTARIA,PORTARIA,-1,2024,3,Universidade Federal da Paraíba,***.999.934-**_2034264_PEDRO GOMES DE ALMEIDA ...


In [ ]:
df_unique = df.drop_duplicates(subset=['Id_SERVIDOR_PORTAL'])


In [106]:
df_unique

,Id_SERVIDOR_PORTAL,NOME,CPF,MATRICULA,DESCRICAO_CARGO,CLASSE_CARGO,REFERENCIA_CARGO,PADRAO_CARGO,NIVEL_CARGO,SIGLA_FUNCAO,...,DOCUMENTO_INGRESSO_SERVICOPUBLICO,DATA_DIPLOMA_INGRESSO_SERVICOPUBLICO,DIPLOMA_INGRESSO_CARGOFUNCAO,DIPLOMA_INGRESSO_ORGAO,DIPLOMA_INGRESSO_SERVICOPUBLICO,UF_EXERCICIO,ANO,MES,ORG_PADRONIZADO,ID_UNICO
66,1040644,DIEGO ALVES AUGUSTO MARINS,***.743.367-**,021****,AUDITOR,E,0.0,407,0.0,-1,...,2664,14/04/2014,NaN,PORTARIA,PORTARIA,-1,2024,3,Colégio Pedro II,***.743.367-**_1040644_DIEGO ALVES AUGUSTO MARINS
67,1149929,CLAUDIO ANTONIO COSTA DE BARROS,***.497.587-**,016****,CONTADOR,E,0.0,411,0.0,-1,...,2016,13/01/2009,NaN,PORTARIA,PORTARIA,-1,2024,3,Colégio Pedro II,***.497.587-**_1149929_CLAUDIO ANTONIO COSTA D...
68,1371705,CRISTINA ALVES PEREIRA,***.737.987-**,017****,CONTADOR,E,0.0,410,0.0,-1,...,1726,28/09/2009,NaN,PORTARIA,PORTARIA,-1,2024,3,Colégio Pedro II,***.737.987-**_1371705_CRISTINA ALVES PEREIRA
69,374940,FABIO COSTA DE ALMEIDA,***.003.207-**,021****,CONTADOR,E,0.0,407,0.0,-1,...,963,17/03/2014,NaN,PORTARIA,PORTARIA,-1,2024,3,Colégio Pedro II,***.003.207-**_374940_FABIO COSTA DE ALMEIDA
70,1064625,FELLIPE SANTOS COELHO,***.491.127-**,021****,CONTADOR,E,0.0,407,0.0,-1,...,964,17/03/2014,NaN,PORTARIA,PORTARIA,-1,2024,3,Colégio Pedro II,***.491.127-**_1064625_FELLIPE SANTOS COELHO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
714913,262971,THAINARA SILVA ALMEIDA,***.346.605-**,020****,ASSISTENTE EM CIENCIA E TECNOLOGIA,S,0.0,V,0.0,-1,...,114,09/09/2013,NaN,PORTARIA,PORTARIA,-1,2024,3,Fundação Coordenação de Aperfeiçoamento de Pes...,***.346.605-**_262971_THAINARA SILVA ALMEIDA
714915,2445139,THAIS SAUTCHUK PIMENTA,***.082.591-**,016****,ASSISTENTE EM CIENCIA E TECNOLOGIA,R,0.0,III,0.0,-1,...,38,25/05/2009,NaN,PORTARIA,PORTARIA,-1,2024,3,Fundação Coordenação de Aperfeiçoamento de Pes...,***.082.591-**_2445139_THAIS SAUTCHUK PIMENTA
714916,2936633,THIAGO CARVALHO MARTINS,***.317.121-**,017****,ASSISTENTE EM CIENCIA E TECNOLOGIA,S,0.0,V,0.0,-1,...,1175,11/01/2010,NaN,PORTARIA,PORTARIA,-1,2024,3,Fundação Coordenação de Aperfeiçoamento de Pes...,***.317.121-**_2936633_THIAGO CARVALHO MARTINS
714917,1150414,THINAMI SAIKI,***.050.911-**,017****,ASSISTENTE EM CIENCIA E TECNOLOGIA,R,0.0,III,0.0,-1,...,038,25/05/2009,NaN,PORTARIA,PORTARIA,-1,2024,3,Fundação Coordenação de Aperfeiçoamento de Pes...,***.050.911-**_1150414_THINAMI SAIKI


In [ ]:
df_unique['ORGSUP_LOTACAO']

In [ ]:
df_unique[df_unique['ORGSUP_LOTACAO']=='MINISTERIO DE PORTOS E AEROPORTOS'] 
#   "MIN GESTAO E INOV EM SERV PUBLICOS": "MGI - Órgão setorial", 3403 > ver em orgao superior ('MIN GESTAO E INOV EM SERV PUBLICOS') : FINAL É 12913 OK
#    "Agência Nacional do Petróleo, Gás Natural e Biocombustíveis": "Agência Nacional de Petróleo, Gás Natural e Biocombustíveis (ANP)", FINAL É 628 ok
#   "Agência Nacional de Transportes Terrestres": "Agência Nacional de Transportes Terrestres (ANTT)", FINAL É 882 ok
#    "MINISTERIO DA PREVIDENCIA SOCIAL": "Ministério da Previdência Social (MPS)", 3460 > ver em org superior(MINISTERIO DA PREVIDENCIA SOCIAL) FINAL é 18501 OK
#    "MIN DESENV ASSIS SOCI FAMIL COMBATE FOME": "Ministério de Desenvolvimento Social e Combate à Fome (MDS)", FINAL É 213 ok
#    "MINISTERIO DO MEIO AMBIENTE": "Ministério do Meio Ambiente e Mudança do Clima (MMA)", 510  > ver qtd em org superior ('MINISTERIO DO MEIO AMBIENTE') FINAL É 4516 4516 
#    "MINISTERIO DE PORTOS E AEROPORTOS": "Ministério de Portos e Aeroportos (MPOR)", > ver qtd em org superior ('MINISTERIO DE PORTOS E AEROPORTOS') 1504 
#    "MIN EMPREEND MICROEMP EMP PEQUENO PORTE": "Ministério do Empreendedorismo (MEMP)" 11 ok

,Id_SERVIDOR_PORTAL,NOME,CPF,MATRICULA,DESCRICAO_CARGO,CLASSE_CARGO,REFERENCIA_CARGO,PADRAO_CARGO,NIVEL_CARGO,SIGLA_FUNCAO,...,DOCUMENTO_INGRESSO_SERVICOPUBLICO,DATA_DIPLOMA_INGRESSO_SERVICOPUBLICO,DIPLOMA_INGRESSO_CARGOFUNCAO,DIPLOMA_INGRESSO_ORGAO,DIPLOMA_INGRESSO_SERVICOPUBLICO,UF_EXERCICIO,ANO,MES,ORG_PADRONIZADO,ID_UNICO
278837,1871720,LUCIANA CAVALCANTI DE OLIVEIRA SANTANA,***.026.641-**,014****,ANALISTA ADMINISTRATIVO,S,0.0,II,0.0,-1,...,1400,24/11/2004,NaN,PORTARIA,PORTARIA,-1,2024,3,Agência Nacional de Aviação Civil,***.026.641-**_1871720_LUCIANA CAVALCANTI DE O...
278998,3207479,FABIO BARBOSA MARRA,***.886.456-**,015****,ANALISTA ADMINISTRATIVO,S,0.0,II,0.0,-1,...,899,24/08/2007,NaN,PORTARIA,PORTARIA,-1,2024,3,Agência Nacional de Aviação Civil,***.886.456-**_3207479_FABIO BARBOSA MARRA
298657,2540556,LUZIANE MARIA MOURA DE CASTELLO,***.105.403-**,011****,AGENTE ADMINISTRATIVO,S,0.0,III,0.0,-1,...,000000889,30/08/1995,NaN,PORTARIA,PORTARIA,-1,2024,3,Agência Nacional de Transportes Aquaviários,***.105.403-**_2540556_LUZIANE MARIA MOURA DE ...
364077,1138324,SILVIO MARCELINO DE OLIVEIRA FILHO,***.584.658-**,002****,TECNICO,M,0.0,III,0.0,-1,...,000000323,02/04/1984,NaN,PORTARIA,PORTARIA,-1,2024,3,Agência Nacional de Aviação Civil,***.584.658-**_1138324_SILVIO MARCELINO DE OLI...
364196,3377216,MARCO AURELIO DE REZENDE BARRETO,***.812.207-**,002****,ECONOMISTA,S,0.0,III,0.0,-1,...,975/1SDPC,09/07/1981,NaN,PORTARIA,PORTARIA,-1,2024,3,Agência Nacional de Aviação Civil,***.812.207-**_3377216_MARCO AURELIO DE REZEND...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
640839,1634082,TERESA CRISTINA DE CARVALHO PINHEIRO,***.378.256-**,013****,ESP EM REG DE SERV DE TRANSP AQUAVIARIOS,B,0.0,V,0.0,-1,...,070,08/09/2009,NaN,PORTARIA,PORTARIA,-1,2024,3,Agência Nacional de Transportes Aquaviários,***.378.256-**_1634082_TERESA CRISTINA DE CARV...
640841,476589,TIAGO COSTA DE THUIN,***.296.037-**,015****,ESP EM REG DE SERV DE TRANSP AQUAVIARIOS,B,0.0,V,0.0,-1,...,71,30/08/2006,NaN,PORTARIA,PORTARIA,-1,2024,3,Agência Nacional de Transportes Aquaviários,***.296.037-**_476589_TIAGO COSTA DE THUIN
640845,1340164,WAGNER SILVA DE SIQUEIRA,***.773.747-**,022****,ESP EM REG DE SERV DE TRANSP AQUAVIARIOS,B,0.0,IV,0.0,-1,...,73,08/07/2015,NaN,PORTARIA,PORTARIA,-1,2024,3,Agência Nacional de Transportes Aquaviários,***.773.747-**_1340164_WAGNER SILVA DE SIQUEIRA
640847,2451435,WESLEY ALVES MESQUITA,***.415.621-**,023****,ESP EM REG DE SERV DE TRANSP AQUAVIARIOS,B,0.0,II,0.0,-1,...,117,16/05/2017,NaN,PORTARIA,PORTARIA,-1,2024,3,Agência Nacional de Transportes Aquaviários,***.415.621-**_2451435_WESLEY ALVES MESQUITA


In [117]:
df_unique

,Id_SERVIDOR_PORTAL,NOME,CPF,MATRICULA,DESCRICAO_CARGO,CLASSE_CARGO,REFERENCIA_CARGO,PADRAO_CARGO,NIVEL_CARGO,SIGLA_FUNCAO,...,DOCUMENTO_INGRESSO_SERVICOPUBLICO,DATA_DIPLOMA_INGRESSO_SERVICOPUBLICO,DIPLOMA_INGRESSO_CARGOFUNCAO,DIPLOMA_INGRESSO_ORGAO,DIPLOMA_INGRESSO_SERVICOPUBLICO,UF_EXERCICIO,ANO,MES,ORG_PADRONIZADO,ID_UNICO
66,1040644,DIEGO ALVES AUGUSTO MARINS,***.743.367-**,021****,AUDITOR,E,0.0,407,0.0,-1,...,2664,14/04/2014,NaN,PORTARIA,PORTARIA,-1,2024,3,Colégio Pedro II,***.743.367-**_1040644_DIEGO ALVES AUGUSTO MARINS
67,1149929,CLAUDIO ANTONIO COSTA DE BARROS,***.497.587-**,016****,CONTADOR,E,0.0,411,0.0,-1,...,2016,13/01/2009,NaN,PORTARIA,PORTARIA,-1,2024,3,Colégio Pedro II,***.497.587-**_1149929_CLAUDIO ANTONIO COSTA D...
68,1371705,CRISTINA ALVES PEREIRA,***.737.987-**,017****,CONTADOR,E,0.0,410,0.0,-1,...,1726,28/09/2009,NaN,PORTARIA,PORTARIA,-1,2024,3,Colégio Pedro II,***.737.987-**_1371705_CRISTINA ALVES PEREIRA
69,374940,FABIO COSTA DE ALMEIDA,***.003.207-**,021****,CONTADOR,E,0.0,407,0.0,-1,...,963,17/03/2014,NaN,PORTARIA,PORTARIA,-1,2024,3,Colégio Pedro II,***.003.207-**_374940_FABIO COSTA DE ALMEIDA
70,1064625,FELLIPE SANTOS COELHO,***.491.127-**,021****,CONTADOR,E,0.0,407,0.0,-1,...,964,17/03/2014,NaN,PORTARIA,PORTARIA,-1,2024,3,Colégio Pedro II,***.491.127-**_1064625_FELLIPE SANTOS COELHO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
714913,262971,THAINARA SILVA ALMEIDA,***.346.605-**,020****,ASSISTENTE EM CIENCIA E TECNOLOGIA,S,0.0,V,0.0,-1,...,114,09/09/2013,NaN,PORTARIA,PORTARIA,-1,2024,3,Fundação Coordenação de Aperfeiçoamento de Pes...,***.346.605-**_262971_THAINARA SILVA ALMEIDA
714915,2445139,THAIS SAUTCHUK PIMENTA,***.082.591-**,016****,ASSISTENTE EM CIENCIA E TECNOLOGIA,R,0.0,III,0.0,-1,...,38,25/05/2009,NaN,PORTARIA,PORTARIA,-1,2024,3,Fundação Coordenação de Aperfeiçoamento de Pes...,***.082.591-**_2445139_THAIS SAUTCHUK PIMENTA
714916,2936633,THIAGO CARVALHO MARTINS,***.317.121-**,017****,ASSISTENTE EM CIENCIA E TECNOLOGIA,S,0.0,V,0.0,-1,...,1175,11/01/2010,NaN,PORTARIA,PORTARIA,-1,2024,3,Fundação Coordenação de Aperfeiçoamento de Pes...,***.317.121-**_2936633_THIAGO CARVALHO MARTINS
714917,1150414,THINAMI SAIKI,***.050.911-**,017****,ASSISTENTE EM CIENCIA E TECNOLOGIA,R,0.0,III,0.0,-1,...,038,25/05/2009,NaN,PORTARIA,PORTARIA,-1,2024,3,Fundação Coordenação de Aperfeiçoamento de Pes...,***.050.911-**_1150414_THINAMI SAIKI


In [ ]:
df_unique['Id_SERVIDOR_PORTAL'].duplicated(.sum())

0

In [82]:
df_unique[df_unique['ORGSUP_LOTACAO']=='MIN GESTAO E INOV EM SERV PUBLICOS'] 

,Id_SERVIDOR_PORTAL,NOME,CPF,MATRICULA,DESCRICAO_CARGO,CLASSE_CARGO,REFERENCIA_CARGO,PADRAO_CARGO,NIVEL_CARGO,SIGLA_FUNCAO,...,DATA_INGRESSO_ORGAO,DOCUMENTO_INGRESSO_SERVICOPUBLICO,DATA_DIPLOMA_INGRESSO_SERVICOPUBLICO,DIPLOMA_INGRESSO_CARGOFUNCAO,DIPLOMA_INGRESSO_ORGAO,DIPLOMA_INGRESSO_SERVICOPUBLICO,UF_EXERCICIO,ANO,MES,ORG_PADRONIZADO
12424,2783973,ELIEZER COSTA DOS SANTOS,***.859.417-**,030****,MOTORISTA - NA,S,0.0,III,0.0,-1,...,19/01/2018,NI,13/02/1986,NaN,PORTARIA,CONTRATO,-1,2024,3,Governo do Ex-Território Federal de Rondônia
12572,497421,DARCY LIMA BARRETO,***.340.052-**,024****,AGENTE ADMINISTRATIVO,S,0.0,III,0.0,-1,...,31/03/2017,NI,20/09/1984,NaN,PORTARIA,CONTRATO,-1,2024,3,Governo do Ex-Território Federal de Rondônia
13896,2265294,DARCLEY DE LIMA ANDRADE,***.390.082-**,029****,AUXILIAR OPERACIONAL SERV DIVERSOS - NA,S,0.0,III,0.0,-1,...,09/06/2017,NI,01/11/1983,NaN,PORTARIA,CONTRATO,-1,2024,3,Governo do Ex-Território Federal de Rondônia
13897,2808714,WALMIR MIRANDA DE ARAUJO,***.977.472-**,023****,AUXILIAR OPERACIONAL SERV DIVERSOS - NA,S,0.0,III,0.0,-1,...,03/02/2017,NI,19/09/1984,NaN,PORTARIA,CONTRATO,-1,2024,3,Governo do Ex-Território Federal de Rondônia
14789,1232533,HUDSON ROMERIO MORAIS DA SILVA GUIMARAES,***.553.152-**,033****,AGENTE ADMINISTRATIVO,S,0.0,III,0.0,-1,...,31/07/2023,8018,31/07/2023,NaN,PORTARIA,CONTRATO,-1,2024,3,Governo do Ex-Território Federal de Roraima
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
683884,434777,MARIA LUCIA LEAL SANTOS,***.706.631-**,023****,ARQUITETO,S,0.0,III,0.0,-1,...,09/06/2016,NI,22/02/1983,NaN,PORTARIA,CONTRATO,-1,2024,3,Governo do Ex-Território Federal de Rondônia
709557,44302,SEBASTIANA SOCORRO DA SILVA ALMEIDA,***.100.692-**,011****,ENGENHEIRO FLORESTAL,S,0.0,III,0.0,-1,...,26/05/2017,NI,29/06/1984,NaN,PORTARIA,CONTRATO,-1,2024,3,Governo do Ex-Território Federal de Rondônia
709560,3526054,ALMIRA FRANCISCO DOS SANTOS CARDOSO,***.296.929-**,023****,AGENTE ADMINISTRATIVO,S,0.0,III,0.0,-1,...,21/07/2016,NI,01/02/1984,NaN,PORTARIA,CONTRATO,-1,2024,3,Governo do Ex-Território Federal de Rondônia
710415,3353081,JOSMARA PEREIRA GOMES,***.655.022-**,030****,AUXILIAR OPERACIONAL SERV DIVERSOS - NA,S,0.0,III,0.0,-1,...,10/11/2017,NI,15/06/1986,NaN,PORTARIA,CONTRATO,-1,2024,3,Governo do Ex-Território Federal de Rondônia


In [ ]:
Ministério da Educação', 'Ministério da Defesa',
       'MIN GESTAO E INOV EM SERV PUBLICOS',
       'MINISTERIO DOS TRANSPORTES', 'Ministério de Minas e Energia',
       'MINISTERIO DA PREVIDENCIA SOCIAL', 'MINISTERIO DA FAZENDA',
       'MINISTERIO DO TRABALHO E EMPREGO', 'Ministério da Saúde',
       'MINISTERIO DO PLANEJAMENTO E ORCAMENTO',
       'MIN DESENVOLV IND COMERCIO E SERVICOS',
       'Ministério da Ciência, Tecnologia, Inovações e Comunicações',
       'MIN DA INTEG E DO DESENV REGIONAL', 'MINISTERIO DO MEIO AMBIENTE',
       'MINISTERIO DOS POVOS INDIGENAS',
       'Ministério das Relações Exteriores',
       'MIN DO DESENV AGR E AGRIC FAMILIAR',
       'MINISTERIO DE PORTOS E AEROPORTOS

In [50]:

# cria dataframe apenas com os órgãos do mapeamento
df_orgaos = (
    df_unique[df_unique['ORG_PADRONIZADO'].isin(mapeamento.values())]  # mantém só os que estão no mapeamento
    .groupby('ORG_PADRONIZADO')
    .size()
    .reset_index(name='Servidores')
)

df_orgaos

,ORG_PADRONIZADO,Servidores
0,Agência Nacional de Transportes Terrestres (ANTT),882
1,MGI - Órgão setorial,3403
2,Ministério da Previdência Social (MPS),3460
3,Ministério de Desenvolvimento Social e Combate...,213
4,Ministério de Portos e Aeroportos (MPOR),49
5,Ministério do Empreendedorismo (MEMP),11
6,Ministério do Meio Ambiente e Mudança do Clima...,510


In [ ]:
df_orgaos['Servidores'].sum()